# Integrate MCP Server's into AgentCore Gateway

## Overview

In large enterprises with thousands of development teams, each running multiple MCP servers with several tools, prompts, and resources, managing and discovering capabilities becomes a critical challenge:

* **Capability Discovery and Sharing**: Teams struggle to discover and consume tools, prompts, and resources across the organization. Should each team maintain separate gateways? How do you share gateway URLs and manage central registries without creating operational overhead?
* **Gateway Management**: Maintaining separate gateways for each MCP server quickly becomes unmanageable at scale.
* **Authentication Complexity**: Managing authentication and authorization across multiple MCP servers becomes increasingly complex, especially with sensitive enterprise data.
* **Maintenance Overhead**: Keeping up with MCP specification updates requires continuous rework and testing across all implementations.

AgentCore Gateway supports onboarding pre-existing MCP server implementations as targets and forwards all three MCP primitive types — **tools**, **prompts**, and **resources** (including URI-templated resources) — to those targets. This tutorial demonstrates the concept end-to-end using FastMCP servers hosted on AgentCore Runtime.

![Diagram](./images/mcp-server-target.png)

## Workshop roadmap

| Step | What you do |
|---|---|
| **1** | Set up the notebook environment (env vars, utilities, logging). |
| **2** | Create the AgentCore Gateway: Cognito inbound auth, IAM role, then the Gateway itself. |
| **3** | Deploy a FastMCP server (with tools, prompts, resources, and a templated resource) to AgentCore Runtime. |
| **4** | Wire that MCP Server in as a Gateway target (outbound OAuth, target creation, verification, inbound token). |
| **5** | Use **tools** through the Gateway with a Strands agent that invokes a tool via MCP. |
| **6** | Use **prompts** through the Gateway: `prompts/list` and `prompts/get`. |
| **7** | Use **resources** through the Gateway: `resources/list`, `resources/read`, and `resources/templates/list`. |
| **8** | Deploy a *second* MCP server with an overlapping resource URI and demonstrate cross-target conflict resolution via the `resourcePriority` field. |
| **9** | Clean up. |

## Tutorial Details

| Information          | Details                                                                              |
|:---------------------|:-------------------------------------------------------------------------------------|
| Tutorial type        | Interactive                                                                          |
| AgentCore components | AgentCore Gateway, AgentCore Identity, AgentCore Runtime                             |
| Agentic Framework    | Strands Agents                                                                       |
| Gateway Target type  | MCP server                                                                           |
| MCP primitives       | Tools, Prompts, Resources (static and templated)                                     |
| Inbound Auth IdP     | Amazon Cognito, but can use others                                                   |
| Outbound Auth        | Amazon Cognito, but can use others                                                   |
| LLM model            | Anthropic Claude Haiku 4.5                                                           |
| Tutorial components  | Creating AgentCore Gateway, invoking tools/prompts/resources, resourcePriority demo  |
| Tutorial vertical    | Cross-vertical                                                                       |
| Example complexity   | Easy                                                                                 |
| SDK used             | boto3                                                                                |

### Step 1: Setup & Prerequisites

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import os

# Set AWS credentials if not using SageMaker notebook
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

# Set specific logger levels
logging.getLogger("strands").setLevel(logging.INFO)

### Step 2: Create AgentCore Gateway

### Step 2.1: Setup Cognito to manage Inbound Auth to AgentCore Gateway

In this demo we are using Amazon Cognito to manage inbound authentication for AgentCore Gateway, however for your enterprise needs you can configure any OAuth 2.0 compliant IDP. Inbound configuration managed who is allowed to invoke the AgentCore Gateway. Checkout [setup](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity-idp-microsoft.html) steps for Okta, Entra ID, Auth, etc.

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(
    cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
print(gw_cognito_discovery_url)

### Step 2.2: Create AgentCore Gateway IAM Role

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-mcpgateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

### Step 2.3: Create AgentCore Gateway

In [ ]:
# CreateGateway with Cognito authorizer. Use the Cognito user pool created in the previous step
import boto3

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            gw_client_id
        ],  # Client MUST match with the ClientId configured in Cognito. Example: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": gw_cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="ac-gateway-mcp-server",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # The IAM Role must have permissions to create/list/get/delete Gateway
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {"supportedVersions": ["2025-11-25"], "searchType": "SEMANTIC"}
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target",
)
print(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

### Step 3: Deploy MCP Server on AgentCore Runtime

### Step 3.1: Setup Cognito to manage Inbound Auth to MCP Server (running on AgentCore Runtime)

This section highlights the ability for the gateway to have a separate inbound authentication from its target systems. This allows agents to access tools that use multiple identity providers through a single interface.

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-runtime-pool"
RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
CLIENT_NAME = "sample-agentcore-runtime-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
runtimeScopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(
    cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
)
print("Resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(
    cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {runtime_client_id}")

# Get discovery URL
runtime_cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration"
print(runtime_cognito_discovery_url)

### Step 3.2: Write the MCP server (tools, prompts, resources, and a templated resource)

You can replace below with your MCP server that might already exist. This section follows the example in the [Hosting MCP Server on Amazon Bedrock AgentCore Runtime](https://github.com/awslabs/amazon-bedrock-agentcore-samples/blob/main/01-tutorials/01-AgentCore-runtime/02-hosting-MCP-server/hosting_mcp_server.ipynb) tutorial.

In [ ]:
%%writefile mcp_server.py
import json

from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


# --- Tools ---------------------------------------------------------------

@mcp.tool()
def getOrder() -> int:
    """Get an order."""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update an existing order."""
    return 456


# --- Prompts -------------------------------------------------------------

@mcp.prompt()
def order_summary_prompt(orderId: int) -> str:
    """Prompt template that asks an LLM to summarize a single order."""
    return f"Summarize the activity on order {orderId}."


@mcp.prompt()
def cancellation_email_prompt(orderId: int, reason: str) -> str:
    """Prompt template for drafting a customer cancellation email."""
    return f"Draft a customer email cancelling order {orderId}. Reason: {reason}."


# --- Resources (static) --------------------------------------------------

@mcp.resource("orders://catalog")
def order_catalog() -> str:
    """Static catalog of orders the server knows about."""
    return json.dumps({
        "orders": [
            {"id": 123, "customer": "alice", "total": 42.00},
            {"id": 456, "customer": "bob", "total": 99.50},
        ]
    })


@mcp.resource("exa://tools/list")
def shadowed_exa_tools_list() -> str:
    """Intentionally collides with the Exa MCP server's `exa://tools/list` resource.

    Step 8 adds Exa as a second gateway target with `resourcePriority=100`. This
    target was created in Step 4.2 with `resourcePriority=10`, so it wins the
    collision — `resources/read exa://tools/list` returns this string rather
    than Exa's real tool catalog.
    """
    return "Shadowed by mcp_server.py — runtime resourcePriority (10) wins over Exa (100)."


# --- Resources (templated) ----------------------------------------------

@mcp.resource("orders://{orderId}/details")
def order_details(orderId: str) -> str:
    """Templated resource: details for a specific order ID."""
    return json.dumps({"orderId": orderId, "status": "shipped", "carrier": "UPS"})


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 3.3: Configure and launch on AgentCore Runtime

Configure the AgentCore Runtime environment for this MCP server, then launch it.

In [ ]:
from runtime_deploy import deploy_mcp_server

deployed = deploy_mcp_server(
    entrypoint="mcp_server.py",
    agent_name="ac_gateway_mcp_server",
    region=REGION,
    runtime_client_id=runtime_client_id,
    runtime_discovery_url=runtime_cognito_discovery_url,
)

agentcore_runtime = deployed.runtime
mcp_arn = deployed.agent_arn
mcp_id = deployed.agent_id
mcp_url = deployed.agent_url

### Step 4: Wire the MCP Server in as a Gateway Target

### Step 4.1: Configure outbound auth (OAuth2 credential provider)

Create an AgentCore Identity Resource Credential Provider that the Gateway will use as outbound auth when it calls the MCP server in AgentCore Runtime.

In [ ]:
import boto3

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name="ac-gateway-mcp-server-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {
                "discoveryUrl": runtime_cognito_discovery_url,
            },
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### Step 4.2: Create the Gateway Target

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_url,
                "resourcePriority": 10,  # lower wins on resource URI collisions; matters in Step 8
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### Step 4.3: Verify the Gateway Target is READY

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

### Step 4.4: Get an inbound access token

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propagation completes"
)
token_response = utils.get_token(
    gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
)
token = token_response["access_token"]
print("Token response:", token)

## Step 5: Use tools through the Gateway

Gateway exposes the MCP server's tools via standard MCP `tools/list` and `tools/call` calls. Below we cover the cached-vs-live behaviour first, then run a Strands agent that invokes a tool through the Gateway.

### Step 5.1: How tools flow through the Gateway

**ListTools (`tools/list`).** Gateway provides access to tool definitions previously synchronized from MCP targets, following a cache-first approach. When a client calls `tools/list`, Gateway returns its cached, normalized tool definitions rather than calling the MCP server live. The cache is populated by **implicit synchronization** during target creation/update, or by **explicit synchronization** via `SynchronizeGatewayTargets`. (See the next notebook, `02-mcp-target-synchronization.ipynb`, for details — including `listingMode='DYNAMIC'`, which forwards `tools/list` live to the MCP server.)

**InvokeTool (`tools/call`).** Tool invocation is *always* live — Gateway opens a session with the MCP server, retrieves fresh outbound credentials from AgentCore Identity if needed, and forwards the call. Tool names sent by the client must include the target prefix `{targetName}___{toolName}` (triple underscore).

![List](images/mcp-server-list-tools.png)

![Invoke](images/mcp-server-invoke-tool.png)

For more relevant examples on AgentCore Gateway search functionality, see [03-search-tools](../03-search-tools).

### Step 5.2: Strands agent invokes a tool via the Gateway

In [ ]:
import json

import requests
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def get_token():
    token = utils.get_token(
        gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
    )
    return token["access_token"]


def create_streamable_http_transport():
    return streamablehttp_client(
        gatewayURL, headers={"Authorization": f"Bearer {get_token()}"}
    )


client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # may need to update model_id depending on region
    temperature=0.7,
    max_tokens=500,  # Limit response length
)

with client:
    # Call the listTools
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(
        model=yourmodel, tools=tools
    )  ## you can replace with any model you like
    # Invoke the agent with the sample prompt. This will only invoke MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    agent("Hi, can you list all tools available to you")
    # Simplified prompt and error handling
    result = agent("Update order 123")

## Step 6: Use prompts through the Gateway

Prompts are parameterized message templates the MCP server exposes for AI generation. Gateway forwards two MCP methods:

- `prompts/list` — served from Gateway's catalog (cached during synchronization) when the target uses the default `listingMode='DEFAULT'`. Prompt names are returned with the target prefix `{targetName}___{promptName}` (triple underscore — same convention as tools).
- `prompts/get` — *always* proxied live to the downstream MCP server, regardless of `listingMode`. The prompt `name` argument MUST include the `targetName___` prefix.

### Step 6.1: `prompts/list` — list prompts available through the Gateway

In [ ]:
import json

from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(
        gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
    )["access_token"]


mcp = GatewayMCPClient(gatewayURL, _get_inbound_token)

print(json.dumps(mcp.list_prompts(), indent=2))

### Step 6.2: `prompts/get` — fetch a rendered prompt

In [ ]:
result = mcp.get_prompt(
    name="mcp-server-target___order_summary_prompt",
    arguments={"orderId": "123"},
)
print(json.dumps(result, indent=2))

## Step 7: Use resources through the Gateway

Resources are addressable content the MCP server exposes via URIs. Templated resources use [RFC 6570](https://datatracker.ietf.org/doc/html/rfc6570) URI templates so a single handler can serve many concrete URIs. Gateway forwards three MCP methods:

- `resources/list` and `resources/templates/list` — served from Gateway's catalog under `listingMode='DEFAULT'`; forwarded live under `listingMode='DYNAMIC'`. Resource URIs are returned **as-is** — there is no `target___` prefix; the original URI from the MCP server is passed through unchanged.
- `resources/read` — *always* proxied live to the downstream MCP server, regardless of `listingMode`.

### Step 7.1: `resources/list` and `resources/read`

> **Security warning**
>
> Resource URIs are provided by the downstream MCP server target and are not validated or sanitized by the gateway. A malicious or compromised MCP server could return URIs pointing to internal endpoints (SSRF) or local filesystem paths (for example, `file:///etc/passwd`). Validate and sanitize resource URIs before following them, and do not automatically fetch or render URIs from untrusted MCP server targets.

In [ ]:
print("--- resources/list ---")
print(json.dumps(mcp.list_resources(), indent=2))

print("\n--- resources/read orders://catalog ---")
print(json.dumps(mcp.read_resource("orders://catalog"), indent=2))

### Step 7.2: `resources/templates/list` and reading a templated URI

Resource templates use [RFC 6570 URI templates](https://datatracker.ietf.org/doc/html/rfc6570) (e.g. `orders://{orderId}/details`). Clients fill in the parameters to produce a concrete URI, which a standard MCP server then serves via `resources/read`.

The first call below — `resources/templates/list` — works as expected and returns the template the upstream server registered.

The second call — `resources/read` against the concrete URI `orders://123/details` derived from that template.

In [ ]:
print("--- resources/templates/list ---")
print(json.dumps(mcp.list_resource_templates(), indent=2))

print("\n--- resources/read orders://123/details ---")
print(json.dumps(mcp.read_resource("orders://123/details"), indent=2))

## Step 8: Multi-target conflict resolution with `resourcePriority`

Tools and prompts are auto-namespaced with `{targetName}___{name}`, so collisions across targets are impossible. Resources are different — Gateway returns resource URIs raw. When two targets expose the same URI, Gateway resolves the read by routing to the target with the lowest `resourcePriority` (range `0–1000`; default `1000`; lower wins).

In this step we add the public **Exa MCP server** (`https://mcp.exa.ai/mcp`) as a *second* gateway target with `resourcePriority=100` — no extra AgentCore Runtime to spin up. Then we list Exa's resources, pick one, and **shadow** it by adding the same URI to `mcp_server.py` (which already has `resourcePriority=10` from Step 4.2). Lower wins, so the runtime server's content takes precedence over Exa's for that URI.

### Step 8.1: Add Exa MCP server as a second gateway target with `resourcePriority=100`

`https://mcp.exa.ai/mcp` is a public managed MCP server (Exa search). Plug it straight in as a target — the gateway is just as happy fronting third-party MCP services as it is your own runtime. Exa exposes one resource, `exa://tools/list`, which the runtime server already shadows (Step 3.2 of `mcp_server.py`). Once Exa is wired in, both targets serve that URI and `resourcePriority` decides who answers the read.

Exa's `resources/list` and `resources/read` are reachable without authentication, so this demo runs without an API key. To exercise Exa's *tools* (`web_search_exa`, `web_fetch_exa`, etc.) you'd need to set `EXA_API_KEY` and pass it as the `exaApiKey` query parameter on the endpoint URL — the next cell does this conditionally. For production use, prefer setting up an **API key credential provider** in AgentCore Identity rather than baking the key into the URL.

In [ ]:
create_exa_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target-exa",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": "https://mcp.exa.ai/mcp",
                "resourcePriority": 100,  # higher than the runtime's 10 — runtime wins on collision
            }
        }
    },
)
exaTargetID = create_exa_target_response["targetId"]
print(f"Created Exa target: {exaTargetID}")

### Step 8.2: Inspect the merged resource catalog

Call `resources/list` through the gateway. The response now includes resources from **both** targets — the runtime server's `orders://...` plus Exa's `exa://tools/list`. You'll see `exa://tools/list` listed **twice** because both targets advertise it.

In [ ]:
print(json.dumps(mcp.list_resources(), indent=2))

### Step 8.3: Read the colliding URI — runtime wins

Both targets expose `exa://tools/list` with different content — the runtime returns a fixed string from `mcp_server.py`, Exa returns its actual tool catalog (web_search_exa, web_fetch_exa, etc.). The runtime target was created with `resourcePriority=10`; Exa with `resourcePriority=100`. Lower wins, so the read should return the runtime's `"Shadowed by mcp_server.py — ..."` string.

> From `gateway-using-mcp-resources-read.html`: *"When multiple targets expose the same resource URI, the gateway routes the request to the target with the lowest `resourcePriority` value."*

In [ ]:
print(json.dumps(mcp.read_resource("exa://tools/list"), indent=2))

### Step 8.4: Naming and conflict-resolution recap

| Capability | Naming across targets | Conflict resolution |
|---|---|---|
| Tools | `targetName___toolName` (triple underscore) | impossible — names are namespaced |
| Prompts | `targetName___promptName` (triple underscore) | impossible — names are namespaced |
| Resources | URI returned as-is, no prefix | `resourcePriority` on target (lower wins; default 1000) |
| Resource templates | URI template returned as-is, no prefix | follows `resourcePriority` |

## Step 9: Clean up

Additional resources are also created like IAM role, IAM Policies, Credentials provider, AWS Lambda functions, Cognito user pools, s3 buckets that you might need to manually delete as part of the clean up. This depends on the example you run.

> **NOTE:** if you are moving on to the next notebook, [02-mcp-target-synchronization](02-mcp-target-synchronization.ipynb), we suggest cleaning these resources for the next tutorial.

In [ ]:
## Step 9.1: Delete the Gateway (transitively deletes both targets, including the Exa target)
utils.delete_gateway(gateway_client, gatewayID)

## Step 9.2: Delete the OAuth2 credential provider
identity_client.delete_oauth2_credential_provider(name="ac-gateway-mcp-server-identity")

## Step 9.3: Delete the Runtime
agentcore_runtime.destroy(delete_ecr_repo=True)

## Step 9.5: Delete the local files the starter-toolkit generated
# `.bedrock_agentcore.yaml` caches agent config; `Dockerfile` and
# `.dockerignore` are the build context the toolkit pushes to CodeBuild.
# Removing them lets a fresh run of this notebook reconfigure from scratch.
from pathlib import Path

for fname in (".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"):
    try:
        Path(fname).unlink()
        print(f"✅ {fname} deleted")
    except FileNotFoundError:
        print(f"ℹ️  {fname} not found")

## Step 9.6: Delete the Cognito user pools (gateway + runtime) and the Gateway IAM role
# utils.delete_cognito_user_pool only calls delete_user_pool, which fails if a hosted-UI
# domain exists on the pool — and get_or_create_user_pool does set one up. Tear the
# domain down first, then use the helper.
import boto3

cognito = boto3.client("cognito-idp", region_name=REGION)
for pool_id in (gw_user_pool_id, runtime_user_pool_id):
    pool = cognito.describe_user_pool(UserPoolId=pool_id)["UserPool"]
    if pool.get("Domain"):
        cognito.delete_user_pool_domain(Domain=pool["Domain"], UserPoolId=pool_id)
    utils.delete_cognito_user_pool(pool_id, region=REGION)

# utils.delete_iam_role detaches managed + inline policies and then deletes the role.
utils.delete_iam_role("agentcore-sample-mcpgateway-role")